# Stage 1 (X-ray) — One-Class Learning (Train on NORMAL only)

This notebook implements **one-class anomaly detection** on NIH ChestX-ray14:

- **Training data:** ONLY `No Finding` images (normal)
- **Model:** Convolutional Autoencoder (reconstruction)
- **Anomaly score:** per-image reconstruction error
- **Outputs:**
  - Continuous **normality/anomaly score** for each image
  - Optional **threshold** for "flag abnormal" (calibrated using *only* normal validation scores via percentile)

> This is NOT a diagnostic tool. It is a proof-of-concept research pipeline stage.

In [ ]:
import os, glob, random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix
)

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)

## 1) Paths
Update `dataset_path` to where you stored the NIH ChestX-ray14 KaggleHub dataset.

In [ ]:
# ===== Paths (adjust if needed) =====
dataset_path = "/path/to/local/resource"

csv_path      = os.path.join(dataset_path, "Data_Entry_2017.csv")
trainval_txt  = os.path.join(dataset_path, "train_val_list.txt")
test_txt      = os.path.join(dataset_path, "test_list.txt")

for p in [csv_path, trainval_txt, test_txt]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing file: {p}\n\nFix dataset_path first.")

print('✅ Found CSV and split files')

## 2) Load CSV + NIH splits

In [ ]:
df = pd.read_csv(csv_path)
df["Image Index"] = df["Image Index"].astype(str).str.strip()
df["Finding Labels"] = df["Finding Labels"].astype(str).str.strip()

with open(trainval_txt, 'r') as f:
    trainval_list = set([line.strip() for line in f if line.strip()])
with open(test_txt, 'r') as f:
    test_list = set([line.strip() for line in f if line.strip()])

print('CSV rows:', len(df))
print('Train/Val list:', len(trainval_list))
print('Test list:', len(test_list))

## 3) Map Image Index → PNG path

In [ ]:
# Index PNGs across images_* folders
image_dirs = []
for d in os.listdir(dataset_path):
    if d.startswith('images_'):
        inner = os.path.join(dataset_path, d, 'images')
        if os.path.isdir(inner):
            image_dirs.append(inner)

print('Image folders found:', len(image_dirs))
assert len(image_dirs) > 0, 'No images_*/images folders found under dataset_path'

image_path_map = {}
for img_dir in image_dirs:
    for p in glob.glob(os.path.join(img_dir, '*.png')):
        image_path_map[os.path.basename(p)] = p

df['Path'] = df['Image Index'].map(image_path_map)
missing = df['Path'].isna().sum()
print('Missing images:', missing)
assert missing == 0, 'Some images are missing from path mapping!'

## 4) Build NORMAL-only training set
- Normal = rows where `Finding Labels == "No Finding"`
- Abnormal = everything else

We will:
- Train on **NORMAL train split only**
- Validate threshold using **NORMAL val split only**
- Evaluate on TEST using (Normal vs Abnormal) for ROC/PR and confusion matrix

In [ ]:
# Split into trainval/test via NIH official lists
trainval_df = df[df['Image Index'].isin(trainval_list)].copy()
test_df     = df[df['Image Index'].isin(test_list)].copy()

assert trainval_df['Image Index'].isin(test_list).sum() == 0, 'Leakage: trainval contains test!'

# Normal / abnormal flags
trainval_df['is_normal'] = (trainval_df['Finding Labels'] == 'No Finding').astype(int)
test_df['is_normal']     = (test_df['Finding Labels'] == 'No Finding').astype(int)

print('TrainVal size:', len(trainval_df))
print('TrainVal normals:', int(trainval_df['is_normal'].sum()))
print('Test size:', len(test_df))
print('Test normals:', int(test_df['is_normal'].sum()))

# Create train/val split inside trainval
trainval_df = trainval_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_frac = 0.10
n_val = int(len(trainval_df) * val_frac)
val_df = trainval_df.iloc[:n_val].copy()
train_df = trainval_df.iloc[n_val:].copy()

# Keep ONLY normal images for training & validation
train_norm = train_df[train_df['Finding Labels'] == 'No Finding'].copy()
val_norm   = val_df[val_df['Finding Labels'] == 'No Finding'].copy()

print('Train normals:', len(train_norm))
print('Val normals:', len(val_norm))

# For evaluation we use ALL test (normal + abnormal)
test_eval = test_df.copy()
print('Test eval:', len(test_eval), ' (normals:', int(test_eval['is_normal'].sum()), ', abnormals:', int((1-test_eval['is_normal']).sum()), ')')

def make_ae_ds(df_in, training=False):
    paths = df_in["Path"].values.astype(str)

    ds = tf.data.Dataset.from_tensor_slices(paths)
    if training:
        ds = ds.shuffle(min(len(df_in), 20000), seed=SEED, reshuffle_each_iteration=True)

    def load_img(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = tf.cast(img, tf.float32) / 255.0
        return img

    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x: (x, x), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


def make_eval_ds(df_in):
    paths = df_in["Path"].values.astype(str)
    y = df_in["is_normal"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, y))

    def load_img(path, y):
        img = tf.io.read_file(path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = tf.cast(img, tf.float32) / 255.0
        return img, y

    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds



## 5) tf.data pipeline
Autoencoder input = output, so dataset returns (image, image)

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Optional: limit for quick runs
# train_norm = train_norm.sample(n=20000, random_state=SEED)
# val_norm = val_norm.sample(n=2000, random_state=SEED)

def load_img(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img

def make_ae_ds(df_in, training=False):
    paths = df_in['Path'].values.astype(str)
    ds = tf.data.Dataset.from_tensor_slices(paths)
    if training:
        ds = ds.shuffle(buffer_size=min(len(df_in), 20000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_img, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x: (x, x), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ae_ds(train_norm, training=True)
val_ds   = make_ae_ds(val_norm, training=False)

print('✅ autoencoder datasets ready')

## 6) Convolutional Autoencoder
A compact model that trains fast.

In [ ]:
from tensorflow.keras import layers as L

def build_autoencoder(img_size=256):
    inputs = tf.keras.Input(shape=(img_size, img_size, 3))

    # Encoder
    x = L.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = L.MaxPool2D()(x)
    x = L.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = L.MaxPool2D()(x)
    x = L.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = L.MaxPool2D()(x)

    # Bottleneck
    x = L.Conv2D(256, 3, padding='same', activation='relu')(x)

    # Decoder
    x = L.UpSampling2D()(x)
    x = L.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = L.UpSampling2D()(x)
    x = L.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = L.UpSampling2D()(x)
    x = L.Conv2D(32, 3, padding='same', activation='relu')(x)

    outputs = L.Conv2D(3, 3, padding='same', activation='sigmoid')(x)

    return tf.keras.Model(inputs, outputs)

ae = build_autoencoder(IMG_SIZE)
ae.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.MeanAbsoluteError()
)

ae.summary()

## 7) Train (NORMAL only)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

history = ae.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

## 8) Anomaly score = reconstruction error
We compute per-image MAE:
\[ score = mean(|x - x_hat|) \]
Higher score => more abnormal.

In [ ]:
@tf.function
def batch_scores(x):
    x_hat = ae(x, training=False)
    # per-image mean absolute error
    return tf.reduce_mean(tf.abs(x - x_hat), axis=[1,2,3])

def score_df(df_in):
    paths = df_in['Path'].values.astype(str)
    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(load_img, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)

    scores = []
    for xb in ds:
        sb = batch_scores(xb)
        scores.append(sb.numpy())
    return np.concatenate(scores, axis=0)

# Scores on normal validation set (for threshold calibration)
val_norm_scores = score_df(val_norm)
print('Val normal scores:', val_norm_scores.shape)
print('Val score stats:', np.min(val_norm_scores), np.mean(val_norm_scores), np.max(val_norm_scores))

## 9) Threshold (calibration using NORMAL ONLY)
We set threshold as a high percentile of normal validation scores.

- `PERCENTILE = 95` means: allow ~5% false alarms on normal images.

In [ ]:
PERCENTILE = 95
threshold = float(np.percentile(val_norm_scores, PERCENTILE))
print(f'Threshold @ {PERCENTILE}th percentile (normal-only):', threshold)

## 10) Evaluate on TEST (Normal vs Abnormal)
This does not train on abnormal labels; it only measures how well the normal-trained detector separates them.

In [ ]:
# Compute scores for ALL test images
scores_test = score_df(test_eval)

y_true = (1 - test_eval['is_normal'].values).astype(int)  # 1 = abnormal, 0 = normal

# Metrics with continuous score
auc = roc_auc_score(y_true, scores_test)
ap  = average_precision_score(y_true, scores_test)
print('TEST ROC-AUC:', auc)
print('TEST PR-AUC :', ap)

# Binary decision using normal-only threshold
y_pred = (scores_test >= threshold).astype(int)  # 1 = flagged abnormal
cm = confusion_matrix(y_true, y_pred)
print('Confusion matrix:\n', cm)

tn, fp, fn, tp = cm.ravel()

sens = tp / (tp + fn + 1e-9)  # recall abnormal
spec = tn / (tn + fp + 1e-9)  # specificity for normal
prec = tp / (tp + fp + 1e-9)

print('Sensitivity (abnormal recall):', sens)
print('Specificity (normal specificity):', spec)
print('Precision (abnormal):', prec)

## 11) Figures (poster-ready) — blue style confusion matrix + ROC + PR + score histogram

In [ ]:
# Confusion matrix (Blues)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal(0)', 'Abnormal(1)'])
plt.figure()
disp.plot(values_format='d', cmap='Blues')
plt.title(f'One-Class X-ray (AE) — Confusion Matrix (thr={threshold:.5f})')
plt.tight_layout()
plt.savefig('cxr_oneclass_confusion_matrix.png', dpi=200)
plt.close()

# ROC curve
fpr, tpr, _ = roc_curve(y_true, scores_test)
plt.figure()
plt.plot(fpr, tpr)
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('One-Class X-ray (AE) — ROC')
plt.tight_layout()
plt.savefig('cxr_oneclass_roc.png', dpi=200)
plt.close()

# PR curve
prec_curve, rec_curve, _ = precision_recall_curve(y_true, scores_test)
plt.figure()
plt.plot(rec_curve, prec_curve)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('One-Class X-ray (AE) — Precision-Recall')
plt.tight_layout()
plt.savefig('cxr_oneclass_pr.png', dpi=200)
plt.close()

# Score histogram (normal vs abnormal)
normal_scores = scores_test[y_true == 0]
abn_scores = scores_test[y_true == 1]

plt.figure()
plt.hist(normal_scores, bins=60, alpha=0.6, label='Normal')
plt.hist(abn_scores, bins=60, alpha=0.6, label='Abnormal')
plt.axvline(threshold, linestyle='--', label=f'Threshold (P{PERCENTILE})')
plt.xlabel('Reconstruction error (MAE)')
plt.ylabel('Count')
plt.title('One-Class X-ray (AE) — Score Distribution')
plt.legend()
plt.tight_layout()
plt.savefig('cxr_oneclass_score_hist.png', dpi=200)
plt.close()

print('Saved:')
print(' - cxr_oneclass_confusion_matrix.png')
print(' - cxr_oneclass_roc.png')
print(' - cxr_oneclass_pr.png')
print(' - cxr_oneclass_score_hist.png')

## 12) (Optional) Visual sanity-check: originals vs reconstructions

In [ ]:
# Visualize a few reconstructions
import matplotlib.pyplot as plt

def show_recon_examples(df_in, n=6, title=''):
    sample = df_in.sample(n=min(n, len(df_in)), random_state=SEED)
    paths = sample['Path'].values.astype(str)

    imgs = []
    for p in paths:
        img = load_img(p).numpy()
        imgs.append(img)
    x = np.stack(imgs, axis=0)
    x_hat = ae.predict(x, verbose=0)

    plt.figure(figsize=(12, 4))
    for i in range(len(x)):
        plt.subplot(2, len(x), i+1)
        plt.imshow(x[i])
        plt.axis('off')
        if i == 0:
            plt.ylabel('Original')
        
        plt.subplot(2, len(x), len(x)+i+1)
        plt.imshow(x_hat[i])
        plt.axis('off')
        if i == 0:
            plt.ylabel('Recon')

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# Examples from normal validation
show_recon_examples(val_norm, n=6, title='Normal (No Finding) — Reconstructions')

# Examples from abnormal test
abn = test_eval[test_eval['is_normal'] == 0]
show_recon_examples(abn, n=6, title='Abnormal (Any Finding) — Reconstructions')